59 mins 30.6 secs

## Load libraries

In [7]:
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import STL
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import ParameterGrid
from tqdm import tqdm
from collections import defaultdict

## Config

In [8]:
TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "AreaCode"

TRAIN_START_DATE = pd.Timestamp("2007-04-01")
TRAIN_END_DATE   = pd.Timestamp("2022-03-31")

# rolling CV
ROLLING_TRAIN_WINDOW = 120   # 10 years
ROLLING_VAL_WINDOW   = 12    # 1 year

# ---- feature lists (adjust to match your data) ----
continuous_cols = [
    "AvgNeighbourPrice_lag1",
    "local_I_lag1",
    "area_km2",
    "centroid_x",
    "centroid_y",
    "CoL_distance_km",
    "LA_FE",
    "sdlt_perc_threshold",
    "dwelling_stock_per_1000",
    "population",
    "ashe_weekly",
    "base_rate",
    "claimant_count_prop",
    "planning_decisions_per_1000",
    "planning_granted_prop",
    "rail_station_entry_exit",
    "GDP",
    "CPIH",
    "LA_embed_0",
    "LA_embed_1",
    "LA_embed_2",
    "LA_embed_3",
    "LA_embed_4"
]

categorical_cols = [
    "LMIQuadrantlag1_2.0",
    "LMIQuadrantlag1_3.0",
    "LMIQuadrantlag1_4.0",
    "Region_East of England",
    "Region_London",
    "Region_North East",
    "Region_North West",
    "Region_South East",
    "Region_South West",
    "Region_West Midlands",
    "Region_Yorkshire and The Humber",
]

# ---- lag plan (always 1 & 12; variants with 2–6; optional 24) ----
lag_combinations = [
    [1, 12],
    [1, 2, 12],
    [1, 2, 3, 12],
    [1, 2, 3, 4, 5, 6, 12],
    [1, 12, 24],
    [1, 2, 12, 24],
    [1, 2, 3, 12, 24],
    [1, 2, 3, 4, 5, 6, 12, 24],
]
all_lags = sorted({l for combo in lag_combinations for l in combo})

# GB tuning grid
gb_param_grid = {
    "max_iter": [300, 600],           # uppber bound on number of trees
    "max_depth": [3, 5, None],        # tree depth (None lets it grow based on other params)
    "learning_rate": [0.05, 0.1],     # step size per tree
    "max_leaf_nodes": [31, 63],       # tree complexity
    "min_samples_leaf": [20, 50],     # regularisation
    "l2_regularization": [0.0, 1.0],  # extra shrinkage
}

## Metrics

In [9]:
def mae(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return np.mean(np.abs(y - yhat))

def rmse(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return np.sqrt(np.mean((y - yhat) ** 2))

def smape(y, yhat, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return 100.0 * np.mean(2.0 * np.abs(yhat - y) / (np.abs(y) + np.abs(yhat) + eps))

def mase(y, yhat, y_train, m=12, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    y_train = np.asarray(y_train)
    if len(y_train) <= m:
        return np.nan
    naive_diff = np.abs(y_train[m:] - y_train[:-m])
    scale = np.mean(naive_diff) + eps
    return np.mean(np.abs(y - yhat)) / scale

## Load data

In [10]:
df = pd.read_excel("../../data/full_data.xlsx", parse_dates=[TIME_COL])
df = df.sort_values([ENTITY_COL, TIME_COL])

mask_tv = (df[TIME_COL] >= TRAIN_START_DATE) & (df[TIME_COL] <= TRAIN_END_DATE)
df_tv = df.loc[mask_tv].copy()

# all unique months in train+val window
dates = sorted(df_tv[TIME_COL].unique())

## Rolling STL feature builder

In [11]:
def add_rolling_stl_components(
    df: pd.DataFrame,
    entity_col: str,
    time_col: str,
    target_col: str,
    period: int = 12,
    min_history: int = 24,
    robust: bool = True,
    show_progress: bool = True,
) -> pd.DataFrame:
    """
    Time-safe rolling STL (one-sided).
    For each entity and each time t, fit STL on y[:t] and assign the last component values to time t.

    Outputs columns:
      - stl_trend
      - stl_seasonal
      - stl_resid

    Notes:
    - This is computationally heavier than "fit once on train then extrapolate".
    - It avoids leakage because STL at time t uses only <= t data.
    """
    df = df.sort_values([entity_col, time_col]).copy()
    df["stl_trend"] = np.nan
    df["stl_seasonal"] = np.nan
    df["stl_resid"] = np.nan

    grouped = df.groupby(entity_col, sort=False)
    iterator = grouped if not show_progress else tqdm(grouped, desc="Rolling STL by LA", leave=False)

    for la, sub in iterator:
        sub = sub.sort_values(time_col)
        y = sub[target_col].astype(float).values
        idx = sub.index.values

        # Rolling one-sided STL: start only when we have enough history
        for t in range(min_history - 1, len(y)):
            y_hist = y[: t + 1]

            # Skip if history contains NaNs
            if np.isnan(y_hist).any():
                continue

            try:
                stl = STL(y_hist, period=period, robust=robust)
                res = stl.fit()

                df.loc[idx[t], "stl_trend"] = float(res.trend[-1])
                df.loc[idx[t], "stl_seasonal"] = float(res.seasonal[-1])
                df.loc[idx[t], "stl_resid"] = float(res.resid[-1])

            except Exception:
                # If STL fails for numeric reasons at this t, leave NaNs
                continue

    return df

## Training with STL and rolling CV

In [13]:
# =========================================================
# CV STRUCTURE:
#   outer loop   = folds (STL + all lags computed once per fold)
#   mid loop     = lag_set (subselect lag columns, scale, etc.)
#   inner loop   = RF params (fit, predict, metrics)
# =========================================================

# metrics_store[(lag_tuple, params_key)] = dict of metric lists across folds
metrics_store = {}

# Pre-build list of folds based on dates
fold_specs = []
start_idx = ROLLING_TRAIN_WINDOW
while True:
    train_end_idx = start_idx
    val_start_idx = train_end_idx
    val_end_idx   = val_start_idx + ROLLING_VAL_WINDOW
    if val_end_idx > len(dates):
        break

    train_start = dates[train_end_idx - ROLLING_TRAIN_WINDOW]
    train_end   = dates[train_end_idx - 1]
    val_start   = dates[val_start_idx]
    val_end     = dates[val_end_idx - 1]

    fold_specs.append((train_start, train_end, val_start, val_end))
    start_idx += ROLLING_VAL_WINDOW

print(f"Number of folds: {len(fold_specs)}")

# =========================================================
# MAIN LOOP: FOLDS → (lag_set → params)
# =========================================================
for fold_no, (train_start, train_end, val_start, val_end) in enumerate(fold_specs, start=1):
    print(f"\n=== Fold {fold_no}: Train {train_start:%Y-%m}–{train_end:%Y-%m}, "
          f"Val {val_start:%Y-%m}–{val_end:%Y-%m} ===")
    
    mask_train = (df_tv[TIME_COL] >= train_start) & (df_tv[TIME_COL] <= train_end)
    mask_val   = (df_tv[TIME_COL] >= val_start)   & (df_tv[TIME_COL] <= val_end)

    fold_train = df_tv.loc[mask_train].copy()
    fold_val   = df_tv.loc[mask_val].copy()

    fold_train["is_train"] = True
    fold_val["is_train"]   = False

    combined = pd.concat([fold_train, fold_val], axis=0).sort_values([ENTITY_COL, TIME_COL])

    combined = add_rolling_stl_components(
        combined,
        entity_col=ENTITY_COL,
        time_col=TIME_COL,
        target_col=TARGET_COL,
        period=12,
        min_history=24,
        robust=True,
        show_progress=True,
    )
    combined = combined.sort_values([ENTITY_COL, TIME_COL])

    for lag in all_lags:
        for comp in ["stl_trend", "stl_seasonal", "stl_resid"]:
            combined[f"{comp}_lag{lag}"] = combined.groupby(ENTITY_COL)[comp].shift(lag)

    for lag_set in lag_combinations:
        lag_cols = [f"{comp}_lag{lag}" for comp in ["stl_trend","stl_seasonal","stl_resid"] for lag in lag_set]
        feature_cols = continuous_cols + categorical_cols + lag_cols

        fold_train_lag = combined[combined["is_train"]].dropna(subset=feature_cols + [TARGET_COL]).copy()
        fold_val_lag   = combined[~combined["is_train"]].dropna(subset=feature_cols + [TARGET_COL]).copy()

        if fold_train_lag.empty or fold_val_lag.empty:
            continue

        # optional for GB (not required)
        scale_cols = continuous_cols + lag_cols
        scaler = StandardScaler()
        fold_train_lag.loc[:, scale_cols] = scaler.fit_transform(fold_train_lag[scale_cols])
        fold_val_lag.loc[:, scale_cols]   = scaler.transform(fold_val_lag[scale_cols])

        X_train, y_train = fold_train_lag[feature_cols], fold_train_lag[TARGET_COL].values
        X_val, y_val     = fold_val_lag[feature_cols], fold_val_lag[TARGET_COL].values

        for params in ParameterGrid(gb_param_grid):
            print(f"  GB params: {params}")
            # make a hashable key for this (lag_set, params)
            lag_key = tuple(lag_set)
            params_key = tuple(sorted(params.items()))
            key = (lag_key, params_key)

            if key not in metrics_store:
                metrics_store[key] = {
                    "lag_set": lag_key,
                    "params": params,
                    "mae": [],
                    "rmse": [],
                    "smape": [],
                    "mase": [],
                    "folds": 0,
                }

            gb = HistGradientBoostingRegressor(
                **params,
                early_stopping=True,
                validation_fraction=0.1,
                n_iter_no_change=10,
                tol=1e-4,
                random_state=42
            )
            gb.fit(X_train, y_train)
            y_pred = gb.predict(X_val)

            metrics_store[key]["mae"].append(mae(y_val, y_pred))
            metrics_store[key]["rmse"].append(rmse(y_val, y_pred))
            metrics_store[key]["smape"].append(smape(y_val, y_pred))
            metrics_store[key]["mase"].append(mase(y_val, y_pred, y_train))
            metrics_store[key]["folds"] += 1



Number of folds: 5


C:\Users\slong\AppData\Local\Temp\ipykernel_30108\2760271798.py:76: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.14753469 -1.14753469 -1.14753469 ...  5.21521037  5.21521037
  5.21521037]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  fold_train_lag.loc[:, scale_cols] = scaler.fit_transform(fold_train_lag[scale_cols])
C:\Users\slong\AppData\Local\Temp\ipykernel_30108\2760271798.py:77: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.14753469 -1.14753469 -1.14753469 ...  5.21521037  5.21521037
  5.21521037]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  fold_val_lag.loc[:, scale_cols]   = scaler.transform(fold_val_lag[scale_cols])


  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 300, 'max_leaf_nodes': 31, 'min_samples_leaf': 20}
  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 300, 'max_leaf_nodes': 31, 'min_samples_leaf': 50}
  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 300, 'max_leaf_nodes': 63, 'min_samples_leaf': 20}
  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 300, 'max_leaf_nodes': 63, 'min_samples_leaf': 50}
  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 600, 'max_leaf_nodes': 31, 'min_samples_leaf': 20}
  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 600, 'max_leaf_nodes': 31, 'min_samples_leaf': 50}
  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 600, 'max_leaf_nodes': 63, 'min_samples_leaf': 20}
  GB p

C:\Users\slong\AppData\Local\Temp\ipykernel_30108\2760271798.py:76: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.14753469 -1.14753469 -1.14753469 ...  5.21521037  5.21521037
  5.21521037]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  fold_train_lag.loc[:, scale_cols] = scaler.fit_transform(fold_train_lag[scale_cols])
C:\Users\slong\AppData\Local\Temp\ipykernel_30108\2760271798.py:77: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.14753469 -1.14753469 -1.14753469 ...  5.21521037  5.21521037
  5.21521037]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  fold_val_lag.loc[:, scale_cols]   = scaler.transform(fold_val_lag[scale_cols])


  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 300, 'max_leaf_nodes': 31, 'min_samples_leaf': 20}
  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 300, 'max_leaf_nodes': 31, 'min_samples_leaf': 50}
  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 300, 'max_leaf_nodes': 63, 'min_samples_leaf': 20}
  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 300, 'max_leaf_nodes': 63, 'min_samples_leaf': 50}
  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 600, 'max_leaf_nodes': 31, 'min_samples_leaf': 20}
  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 600, 'max_leaf_nodes': 31, 'min_samples_leaf': 50}
  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 600, 'max_leaf_nodes': 63, 'min_samples_leaf': 20}
  GB p

...

  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 300, 'max_leaf_nodes': 31, 'min_samples_leaf': 20}
  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 300, 'max_leaf_nodes': 31, 'min_samples_leaf': 50}
  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 300, 'max_leaf_nodes': 63, 'min_samples_leaf': 20}
  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 300, 'max_leaf_nodes': 63, 'min_samples_leaf': 50}
  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 600, 'max_leaf_nodes': 31, 'min_samples_leaf': 20}
  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 600, 'max_leaf_nodes': 31, 'min_samples_leaf': 50}
  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 600, 'max_leaf_nodes': 63, 'min_samples_leaf': 20}
  GB p

C:\Users\slong\AppData\Local\Temp\ipykernel_30108\2760271798.py:76: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.3941662  -1.3941662  -1.3941662  ...  3.54934577  3.54934577
  3.54934577]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  fold_train_lag.loc[:, scale_cols] = scaler.fit_transform(fold_train_lag[scale_cols])
C:\Users\slong\AppData\Local\Temp\ipykernel_30108\2760271798.py:77: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.3941662  -1.3941662  -1.3941662  ...  3.54934577  3.54934577
  3.54934577]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  fold_val_lag.loc[:, scale_cols]   = scaler.transform(fold_val_lag[scale_cols])


  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 300, 'max_leaf_nodes': 31, 'min_samples_leaf': 20}
  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 300, 'max_leaf_nodes': 31, 'min_samples_leaf': 50}
  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 300, 'max_leaf_nodes': 63, 'min_samples_leaf': 20}
  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 300, 'max_leaf_nodes': 63, 'min_samples_leaf': 50}
  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 600, 'max_leaf_nodes': 31, 'min_samples_leaf': 20}
  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 600, 'max_leaf_nodes': 31, 'min_samples_leaf': 50}
  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 600, 'max_leaf_nodes': 63, 'min_samples_leaf': 20}
  GB p

C:\Users\slong\AppData\Local\Temp\ipykernel_30108\2760271798.py:76: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.3941662  -1.3941662  -1.3941662  ...  3.54934577  3.54934577
  3.54934577]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  fold_train_lag.loc[:, scale_cols] = scaler.fit_transform(fold_train_lag[scale_cols])
C:\Users\slong\AppData\Local\Temp\ipykernel_30108\2760271798.py:77: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.3941662  -1.3941662  -1.3941662  ...  3.54934577  3.54934577
  3.54934577]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  fold_val_lag.loc[:, scale_cols]   = scaler.transform(fold_val_lag[scale_cols])


  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 300, 'max_leaf_nodes': 31, 'min_samples_leaf': 20}
  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 300, 'max_leaf_nodes': 31, 'min_samples_leaf': 50}
  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 300, 'max_leaf_nodes': 63, 'min_samples_leaf': 20}
  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 300, 'max_leaf_nodes': 63, 'min_samples_leaf': 50}
  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 600, 'max_leaf_nodes': 31, 'min_samples_leaf': 20}
  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 600, 'max_leaf_nodes': 31, 'min_samples_leaf': 50}
  GB params: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 600, 'max_leaf_nodes': 63, 'min_samples_leaf': 20}
  GB p

## Results

In [15]:
rows = []
for key, val in metrics_store.items():
    if val["folds"] == 0:
        continue
    rows.append({
        "model_type": "GradientBoosting",
        "lag_set": val["lag_set"],
        "params": val["params"],
        "folds": val["folds"],
        "MAE_mean":   float(np.mean(val["mae"])),
        "MAE_std":    float(np.std(val["mae"])),
        "RMSE_mean":  float(np.mean(val["rmse"])),
        "RMSE_std":   float(np.std(val["rmse"])),
        "sMAPE_mean": float(np.mean(val["smape"])),
        "sMAPE_std":  float(np.std(val["smape"])),
        "MASE_mean":  float(np.mean(val["mase"])),
        "MASE_std":   float(np.std(val["mase"])),
    })

results_df = pd.DataFrame(rows).sort_values("RMSE_mean").reset_index(drop=True)
print(results_df.head(20))
results_df.to_csv("../../results/gb_leakfree_stl_rollingcv_results.csv", index=False)

          model_type                 lag_set  \
0   GradientBoosting                 (1, 12)   
1   GradientBoosting                 (1, 12)   
2   GradientBoosting           (1, 2, 3, 12)   
3   GradientBoosting           (1, 2, 3, 12)   
4   GradientBoosting                 (1, 12)   
5   GradientBoosting                 (1, 12)   
6   GradientBoosting                 (1, 12)   
7   GradientBoosting                 (1, 12)   
8   GradientBoosting                 (1, 12)   
9   GradientBoosting                 (1, 12)   
10  GradientBoosting          (1, 2, 12, 24)   
11  GradientBoosting          (1, 2, 12, 24)   
12  GradientBoosting  (1, 2, 3, 4, 5, 6, 12)   
13  GradientBoosting  (1, 2, 3, 4, 5, 6, 12)   
14  GradientBoosting           (1, 2, 3, 12)   
15  GradientBoosting           (1, 2, 3, 12)   
16  GradientBoosting          (1, 2, 12, 24)   
17  GradientBoosting          (1, 2, 12, 24)   
18  GradientBoosting             (1, 12, 24)   
19  GradientBoosting             (1, 12,